# Audit données & ML — Stripe Polyglot (Bloc 2)

Notebook de contrôle : vérifie la **qualité des données sources**, la
**cohérence inter-stores** (Postgres → Kafka → Mongo), la **distribution des
features** servies vs entraînées, et la **performance réelle** (servie, pas
resimulée) du modèle de scoring fraude.

Chaque section se termine par des `assert` : ce notebook n'est pas qu'un
rapport, c'est un garde-fou à relancer avant toute démo/soutenance pour
détecter une régression silencieuse.

**Contexte** : le 2026-09-15, ce notebook a servi à diagnostiquer un
incident réel — précision servie tombée à **29%** (voir §4.3) — causé par
une boucle de rétroaction CDC dans `producers/flink_like_job.py` (corrigée
le jour même, commit associé dans l'historique git). §4.3 documente
l'incident en détail avec les chiffres avant/après comme cas d'école.

**Prérequis** : stack Docker up (`make up`), kernel **Stripe Polyglot (bloc 2)**
(`.venv`, cf. `requirements.txt`).


In [1]:
import os, sys, json
from pathlib import Path
from datetime import datetime, timezone, timedelta

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_recall_fscore_support,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import _env  # noqa: F401 — charge .env

import psycopg2
from psycopg2.extras import RealDictCursor
from pymongo import MongoClient
import redis

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)


Project root: /Users/patriceduclos/Library/CloudStorage/GoogleDrive-patrice.noel.duclos@gmail.com/Mon Drive/Certification AI/Bloc2-Architecture-Stripe/Projet bloc 2


In [2]:
# ── Connexions ──────────────────────────────────────────────────────────
PG_CONFIG = dict(
    host=os.environ.get("PG_HOST", "localhost"),
    port=int(os.environ.get("PG_PORT", 5432)),
    dbname=os.environ.get("PG_DB", "stripe_oltp"),
    user=os.environ.get("PG_USER", "stripe_app"),
    password=os.environ.get("PG_PASSWORD", ""),
)
MONGO_URI = (
    f"mongodb://{os.environ.get('MONGO_USER','admin')}:{os.environ.get('MONGO_PASSWORD','')}"
    f"@{os.environ.get('MONGO_HOST','localhost')}:{int(os.environ.get('MONGO_PORT',27017))}/"
)
REDIS_CONFIG = dict(
    host=os.environ.get("REDIS_HOST", "localhost"),
    port=int(os.environ.get("REDIS_PORT", 6379)),
    password=os.environ.get("REDIS_PASSWORD") or None,
    decode_responses=True,
)

def pg_query(sql, params=None):
    with psycopg2.connect(**PG_CONFIG) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
    return pd.DataFrame([dict(r) for r in rows])

mongo = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
mongo_db = mongo[os.environ.get("MONGO_DB", "stripe_nosql")]
r = redis.Redis(**REDIS_CONFIG)

# Vérifie que tout répond avant d'aller plus loin — pas d'intérêt à
# continuer un audit si une des 3 bases est down.
assert pg_query("SELECT 1 AS ok").iloc[0]["ok"] == 1, "Postgres injoignable"
mongo.admin.command("ping")
assert r.ping(), "Redis injoignable"
print("[OK] Postgres, MongoDB, Redis — tous joignables")


[OK] Postgres, MongoDB, Redis — tous joignables


## 1. Qualité des données sources (PostgreSQL)

Contrôles de base attendus de toute donnée avant de faire confiance à un
modèle entraîné dessus : volumétrie, valeurs nulles sur les colonnes
critiques, intégrité référentielle, doublons, valeurs aberrantes.


In [3]:
counts = pg_query('''
    SELECT 'merchants' AS table, COUNT(*) AS n FROM merchants
    UNION ALL SELECT 'customers', COUNT(*) FROM customers
    UNION ALL SELECT 'payment_methods', COUNT(*) FROM payment_methods
    UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
    UNION ALL SELECT 'refunds', COUNT(*) FROM refunds
    UNION ALL SELECT 'fraud_indicators', COUNT(*) FROM fraud_indicators
''')
counts


,table,n
0,merchants,200
1,customers,5000
2,payment_methods,7870
3,transactions,70014
4,refunds,0
5,fraud_indicators,0


In [4]:
# Intégrité référentielle : aucune transaction ne doit pointer vers un
# merchant/customer inexistant (les FK Postgres le garantissent déjà côté
# écriture, mais on le revérifie ici — la garantie vaut ce que vaut le
# schéma, un audit ne doit rien supposer).
orphans = pg_query('''
    SELECT
        SUM(CASE WHEN m.merchant_id IS NULL THEN 1 ELSE 0 END) AS txn_orphan_merchant,
        SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS txn_orphan_customer,
        COUNT(*) AS total_txn
    FROM transactions t
    LEFT JOIN merchants m ON m.merchant_id = t.merchant_id
    LEFT JOIN customers c ON c.customer_id = t.customer_id
''')
display(orphans)
assert orphans.iloc[0]["txn_orphan_merchant"] == 0, "Transactions orphelines (merchant_id invalide)"
assert orphans.iloc[0]["txn_orphan_customer"] == 0, "Transactions orphelines (customer_id invalide)"
print("[OK] Intégrité référentielle transactions -> merchants/customers")


,txn_orphan_merchant,txn_orphan_customer,total_txn
0,0,0,70014


[OK] Intégrité référentielle transactions -> merchants/customers


In [5]:
# Doublons sur idempotency_key : la contrainte UNIQUE en base l'interdit déjà
# (cf. init/postgres/01_ddl.sql) — ce test vérifie que la contrainte existe
# et tient vraiment, pas seulement qu'elle est déclarée dans le SQL source.
dups = pg_query('''
    SELECT idempotency_key, COUNT(*) AS n
    FROM transactions
    GROUP BY idempotency_key
    HAVING COUNT(*) > 1
''')
assert len(dups) == 0, f"{len(dups)} idempotency_key dupliquées !"
print("[OK] Aucune idempotency_key dupliquée")

# Valeurs aberrantes sur amount (BIGINT, centimes) : négatif ou nul suspect,
# et un montant extrême (> 1M€) mérite un coup d'oeil même si pas invalide.
amt = pg_query("SELECT amount FROM transactions")
n_negative = (amt["amount"] <= 0).sum()
n_extreme = (amt["amount"] > 100_000_000).sum()  # > 1M€
print(f"Montants <= 0 : {n_negative} / {len(amt)}")
print(f"Montants > 1M€ : {n_extreme} / {len(amt)}")
assert n_negative == 0, "Des transactions ont un montant nul ou négatif"
fig = px.histogram(amt, x="amount", nbins=80, title="Distribution des montants (centimes, échelle log)",
                    log_y=True)
fig.show()


[OK] Aucune idempotency_key dupliquée


Montants <= 0 : 0 / 70014
Montants > 1M€ : 0 / 70014


In [6]:
# Équilibre des classes (label de vérité terrain du générateur) — un
# déséquilibre extrême ferait planter le calcul de precision/recall sur de
# trop petits effectifs, et un déséquilibre proche de 50/50 serait suspect
# (le taux de fraude visé par seed_data.py/transaction_producer.py est de
# quelques %, cf. FRAUD_RATIO dans .env).
labels = pg_query('''
    SELECT (metadata->>'is_fraud_pattern')::boolean AS is_fraud, COUNT(*) AS n
    FROM transactions
    WHERE metadata->>'is_fraud_pattern' IS NOT NULL
    GROUP BY 1
''')
labels["pct"] = (labels["n"] / labels["n"].sum() * 100).round(2)
display(labels)
fraud_ratio = labels.loc[labels["is_fraud"] == True, "pct"].values[0] / 100
assert 0.005 < fraud_ratio < 0.30, f"Taux de fraude {fraud_ratio:.1%} hors plage plausible"
print(f"[OK] Taux de fraude générateur : {fraud_ratio:.1%}")


,is_fraud,n,pct
0,False,53938,77.03
1,True,16080,22.97


[OK] Taux de fraude générateur : 23.0%


## 2. Cohérence inter-stores (Postgres → Kafka → MongoDB)

Le pipeline dénormalise chaque transaction scorée dans MongoDB
(`transaction_logs`, et `fraud_alerts` si review/block). Ce qui doit être
vrai si le pipeline est sain : **chaque `txn_id` n'apparaît qu'une seule
fois** dans `fraud_alerts` pour une fenêtre temporelle donnée — un doublon
signale une transaction scorée deux fois (exactement le bug diagnostiqué en
§4.3).


In [7]:
pipeline = mongo_db.fraud_alerts.aggregate([
    {"$match": {"created_at": {"$gt": datetime.now(timezone.utc) - timedelta(minutes=30)}}},
    {"$group": {"_id": "$txn_id", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
])
dup_txn = list(pipeline)
print(f"txn_id dupliqués dans fraud_alerts (30 dernières minutes) : {len(dup_txn)}")
if dup_txn:
    display(pd.DataFrame(dup_txn).head(10))
# Garde-fou de non-régression pour le bug du 2026-09-15 (double-scoring via
# echo CDC du write-back) — corrigé dans producers/flink_like_job.py.
# Si ce test échoue de nouveau, le correctif a régressé.
assert len(dup_txn) == 0, (
    "Des txn_id sont scorés plusieurs fois — même symptôme que l'incident du "
    "2026-09-15 (cf. producers/flink_like_job.py, garde-fou 'fraud_score is not None')"
)
print("[OK] Aucun txn_id dupliqué dans fraud_alerts sur les 30 dernières minutes")


txn_id dupliqués dans fraud_alerts (30 dernières minutes) : 0
[OK] Aucun txn_id dupliqué dans fraud_alerts sur les 30 dernières minutes


In [8]:
# Volume relatif : transaction_logs doit contenir ~1 doc par transaction
# récente scorée (INSERT only), fraud_alerts seulement les review/block —
# donc n(fraud_alerts) <= n(transaction_logs) sur la même fenêtre.
window_start = datetime.now(timezone.utc) - timedelta(hours=1)
n_logs = mongo_db.transaction_logs.count_documents({"created_at": {"$gt": window_start}})
n_alerts = mongo_db.fraud_alerts.count_documents({"created_at": {"$gt": window_start}})
n_pg = pg_query("SELECT COUNT(*) AS n FROM transactions WHERE created_at > %s",
                 (window_start,)).iloc[0]["n"]
print(f"Dernière heure — Postgres: {n_pg} · Mongo transaction_logs: {n_logs} · fraud_alerts: {n_alerts}")
assert n_alerts <= n_logs, "Plus d'alertes que de logs bruts : incohérence de pipeline"


Dernière heure — Postgres: 20678 · Mongo transaction_logs: 20679 · fraud_alerts: 15939


## 3. Distribution des features — entraînement vs service

Le modèle est entraîné sur des features recalculées en SQL (vélocité par
sous-requête corrélée, cf. `ml/train_fraud_model.py::extract_training_data`)
mais servies en production depuis Redis
(`producers/flink_like_job.py::score_transaction`). Si les deux
distributions divergent (« train/serve skew »), le modèle voit en
production des valeurs qu'il n'a jamais vues à l'entraînement — c'est la
cause racine de l'incident du 2026-09-15 (vélocité Redis doublée par un bug
de double-scoring, cf. §4.3).


In [9]:
sys.path.insert(0, str(PROJECT_ROOT))
from ml.features import build_feature_vector, FEATURE_NAMES

# Vélocité "vérité" recalculée en SQL sur les transactions des 20 dernières minutes
# (même requête que extract_current_window() dans ml/train_fraud_model.py,
# reprise ici pour ne pas dépendre d'un import qui exécute tout le module).
served = pg_query('''
    SELECT t.txn_id, t.customer_id, t.amount, t.created_at, t.ip_country, t.device_type,
           t.fraud_score, t.velocity_1h AS served_v1h, t.velocity_24h AS served_v24h
    FROM transactions t
    WHERE t.created_at > NOW() - INTERVAL '20 minutes' AND t.fraud_score IS NOT NULL
''') if False else None
# velocity_1h/24h ne sont pas persistées dans `transactions` (seulement dans
# fraud_alerts/ml_features Mongo) — on les récupère depuis fraud_alerts pour
# les transactions review/block, seul sous-ensemble où elles sont écrites.
served_docs = list(mongo_db.fraud_alerts.find(
    {"created_at": {"$gt": datetime.now(timezone.utc) - timedelta(minutes=20)}},
    {"txn_id": 1, "velocity_1h": 1, "velocity_24h": 1, "customer_id": 1, "created_at": 1, "_id": 0},
))
served_v = pd.DataFrame(served_docs)
if len(served_v):
    fig = px.histogram(served_v, x="velocity_1h", nbins=40,
                        title="Vélocité 1h SERVIE (Redis, via fraud_alerts) — 20 dernières minutes")
    fig.show()
    print(served_v[["velocity_1h", "velocity_24h"]].describe())
else:
    print("Pas assez d'alertes récentes pour tracer la distribution — relancer plus tard.")


       velocity_1h  velocity_24h
count  3063.000000   3063.000000
mean      8.073457     14.861900
std       7.317884     16.434763
min       1.000000      1.000000
25%       1.000000      1.000000
50%       7.000000      9.000000
75%      13.000000     25.000000
max      47.000000     85.000000


In [10]:
# Vélocité "entraînement" : même fenêtre, recalculée en SQL pur (COUNT des
# transactions précédentes du même client dans les 1h/24h qui précèdent),
# strictement indépendante de l'état Redis.
trained_v = pg_query('''
    SELECT
        t.txn_id,
        (SELECT COUNT(*) FROM transactions t2
         WHERE t2.customer_id = t.customer_id
           AND t2.created_at BETWEEN t.created_at - INTERVAL '1 hour' AND t.created_at) AS train_v1h,
        (SELECT COUNT(*) FROM transactions t2
         WHERE t2.customer_id = t.customer_id
           AND t2.created_at BETWEEN t.created_at - INTERVAL '24 hours' AND t.created_at) AS train_v24h
    FROM transactions t
    WHERE t.created_at > NOW() - INTERVAL '20 minutes' AND t.fraud_score IS NOT NULL
    ORDER BY random()
    LIMIT 2000
''')
fig = px.histogram(trained_v, x="train_v1h", nbins=40,
                    title="Vélocité 1h ENTRAÎNEMENT (SQL, recalculée) — même fenêtre, échantillon 2000")
fig.show()
print(trained_v[["train_v1h", "train_v24h"]].describe())

# Comparaison des moyennes : un facteur > 1.5x est le signal exact de
# l'incident du 2026-09-15 (vélocité servie ~2x la vélocité réelle à cause
# du double-scoring). Seuil large car les deux échantillons ne portent pas
# exactement sur les mêmes transactions (fraud_alerts = review/block only).
if len(served_v):
    ratio = served_v["velocity_1h"].mean() / max(trained_v["train_v1h"].mean(), 0.01)
    print(f"\nRatio moyenne(vélocité servie) / moyenne(vélocité entraînement) = {ratio:.2f}")
    if ratio > 1.5:
        print("[ALERTE] Skew significatif — vélocité servie très supérieure à la vélocité réelle.")
    else:
        print("[OK] Pas de skew significatif détecté sur cette fenêtre.")


         train_v1h  train_v24h
count  2000.000000  2000.00000
mean      7.542500    19.06600
std       4.898284     9.55805
min       1.000000     1.00000
25%       4.000000    12.00000
50%       7.000000    18.00000
75%       9.000000    24.00000
max      40.000000    85.00000

Ratio moyenne(vélocité servie) / moyenne(vélocité entraînement) = 1.07
[OK] Pas de skew significatif détecté sur cette fenêtre.


## 4. Performance réelle du modèle (servie, pas resimulée)

**Principe non négociable** (cf. `ml/monitor.py::compute_served_performance`) :
on mesure la précision/rappel sur les **décisions réellement prises et
enregistrées** par le scorer en production (`transactions.fraud_score`
comparé à `metadata->>'is_fraud_pattern'`), jamais en resimulant le scoring
via SQL — une resimulation ne peut pas détecter un bug de la chaîne de
service elle-même (feature store Redis, double-scoring, modèle périmé en
cache...).


In [11]:
def served_precision_recall(since, until=None):
    """Precision/recall sur les décisions réellement servies dans [since, until)."""
    params = [since]
    until_clause = ""
    if until is not None:
        until_clause = "AND created_at < %s"
        params.append(until)
    sql = f'''
        SELECT
            COUNT(*) AS total,
            SUM(CASE WHEN fraud_score >= 0.85 THEN 1 ELSE 0 END) AS predicted_fraud,
            SUM(CASE WHEN (metadata->>'is_fraud_pattern')::boolean THEN 1 ELSE 0 END) AS actual_fraud,
            SUM(CASE WHEN fraud_score >= 0.85 AND (metadata->>'is_fraud_pattern')::boolean
                     THEN 1 ELSE 0 END) AS tp,
            SUM(CASE WHEN fraud_score >= 0.85 AND NOT (metadata->>'is_fraud_pattern')::boolean
                     THEN 1 ELSE 0 END) AS fp,
            SUM(CASE WHEN fraud_score < 0.85 AND (metadata->>'is_fraud_pattern')::boolean
                     THEN 1 ELSE 0 END) AS fn
        FROM transactions
        WHERE created_at > %s {until_clause} AND fraud_score IS NOT NULL
    '''
    row = pg_query(sql, tuple(params)).iloc[0]
    precision = row["tp"] / row["predicted_fraud"] if row["predicted_fraud"] else float("nan")
    recall = row["tp"] / row["actual_fraud"] if row["actual_fraud"] else float("nan")
    return dict(row) | {"precision": precision, "recall": recall}

current_window_start = datetime.now(timezone.utc) - timedelta(minutes=15)
live = served_precision_recall(current_window_start)
print(f"Fenêtre : depuis {current_window_start.isoformat()}")
for k, v in live.items():
    print(f"  {k:16s} = {v}")


Fenêtre : depuis 2026-09-15T14:08:56.690644+00:00
  total            = 4995
  predicted_fraud  = 897
  actual_fraud     = 916
  tp               = 857
  fp               = 40
  fn               = 59
  precision        = 0.955406911928651
  recall           = 0.9355895196506551


### 4.3 — Incident documenté : double-scoring via écho CDC (2026-09-15)

**Chronologie** :
1. `producers/flink_like_job.py` écrit `fraud_score` dans `transactions` après scoring (« write-back »)
2. Debezium capte cet `UPDATE` et le republie sur le **même topic** `stripe.public.transactions`
3. Le consumer (même `group.id`) reconsomme ce message et **rescoie la transaction une 2e fois** — incrémentant la vélocité Redis du client une 2e fois au passage
4. Chaque transaction est donc scorée avec une vélocité en moyenne ~2x sa valeur réelle → train/serve skew massif → énormément de faux positifs

**Mesuré en direct pendant l'investigation** (requêtes SQL identiques à `served_precision_recall()` ci-dessus) :

| Fenêtre | Précision | Rappel | Échantillon |
|---|---|---|---|
| Avant fix (depuis dernier restart, 11:18 UTC) | **29,1%** | 96,2% | 57 884 txns |
| Après fix (`git log` — garde-fou anti-écho CDC) | **95,2%** | 94,6% | 663 txns |

**Correctif** (`producers/flink_like_job.py`, `flink/fraud_scoring_job.py` en miroir) :
un garde `if txn.get("fraud_score") is not None: continue` juste après le parsing du
message, avant tout calcul de vélocité — ignore l'écho de son propre write-back.

**Correctif connexe** (`ml/scoring.py::load_model()`) : le modèle était mis en
cache en mémoire process sans jamais vérifier si le fichier `.pkl` avait été
réécrit par un retrain automatique (`ml/monitor.py`) — un scorer longue durée ne
voyait donc jamais les retrains. Ajout d'un check de `mtime` avant chaque
utilisation du cache.

La cellule suivante recharge cette mesure en direct — si tu relances ce
notebook plus tard, les chiffres seront différents (nouvelle fenêtre) mais
doivent rester dans la même plage que la colonne « Après fix ».


In [12]:
# Seuils issus de ml/monitor.py (ML_MIN_RECALL=0.7, ML_MIN_PRECISION=0.5) —
# mêmes seuils que ceux qui déclenchent une alerte + retrain automatique en
# production, pour que ce notebook et le monitoring en ligne soient d'accord.
ML_MIN_RECALL = float(os.environ.get("ML_MIN_RECALL", 0.7))
ML_MIN_PRECISION = float(os.environ.get("ML_MIN_PRECISION", 0.5))

if live["predicted_fraud"] and live["actual_fraud"]:
    print(f"Précision servie : {live['precision']:.1%}  (seuil mini {ML_MIN_PRECISION:.0%})")
    print(f"Rappel servi     : {live['recall']:.1%}  (seuil mini {ML_MIN_RECALL:.0%})")
    assert live["precision"] >= ML_MIN_PRECISION, (
        f"Précision servie {live['precision']:.1%} sous le seuil {ML_MIN_PRECISION:.0%} — "
        "régression possible du bug de double-scoring, cf. §4.3"
    )
    assert live["recall"] >= ML_MIN_RECALL, f"Rappel servi {live['recall']:.1%} sous le seuil"
    print("[OK] Performance servie dans les clous")
else:
    print("[SKIP] Pas assez de transactions scorées dans la fenêtre pour conclure — relancer plus tard.")


Précision servie : 95.5%  (seuil mini 50%)
Rappel servi     : 93.6%  (seuil mini 70%)
[OK] Performance servie dans les clous


In [13]:
# Matrice de confusion + courbes ROC/PR sur un échantillon avec le score
# continu (pas juste la décision binaire >= 0.85) pour juger la calibration
# du modèle au-delà du seuil opérationnel.
sample = pg_query('''
    SELECT fraud_score, (metadata->>'is_fraud_pattern')::boolean AS y_true
    FROM transactions
    WHERE created_at > %s AND fraud_score IS NOT NULL
''', (current_window_start,))

if len(sample) > 20 and sample["y_true"].nunique() == 2:
    y_true = sample["y_true"].astype(int)
    y_pred = (sample["fraud_score"] >= 0.85).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    fig = px.imshow(cm, text_auto=True, x=["Pred: légitime", "Pred: fraude"],
                     y=["Réel: légitime", "Réel: fraude"], title="Matrice de confusion (servie)",
                     color_continuous_scale="Blues")
    fig.show()
    print(classification_report(y_true, y_pred, target_names=["légitime", "fraude"]))

    fpr, tpr, _ = roc_curve(y_true, sample["fraud_score"])
    auc = roc_auc_score(y_true, sample["fraud_score"])
    prec, rec, _ = precision_recall_curve(y_true, sample["fraud_score"])
    ap = average_precision_score(y_true, sample["fraud_score"])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"ROC (AUC={auc:.3f})"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash"), name="hasard"))
    fig.update_layout(title="Courbe ROC (score continu, servi)", xaxis_title="FPR", yaxis_title="TPR")
    fig.show()

    fig = px.line(x=rec, y=prec, title=f"Courbe Précision/Rappel (AP={ap:.3f})",
                   labels={"x": "Rappel", "y": "Précision"})
    fig.show()
else:
    print("[SKIP] Échantillon trop petit ou classe unique sur cette fenêtre pour les courbes ROC/PR.")


              precision    recall  f1-score   support

    légitime       0.99      0.99      0.99      4079
      fraude       0.96      0.94      0.95       916

    accuracy                           0.98      4995
   macro avg       0.97      0.96      0.97      4995
weighted avg       0.98      0.98      0.98      4995



## 5. Audit du modèle entraîné (artefact + registre)

Contrôle de l'artefact `.pkl` réellement chargé par le scorer en
production : version, features attendues, importance de chaque feature, et
écart entre les métriques **offline** (calculées à l'entraînement,
`.meta.json`) et les métriques **online** mesurées en §4.


In [14]:
import joblib
MODEL_PATH = PROJECT_ROOT / "ml" / "models" / "fraud_xgboost-v1.pkl"
META_PATH = PROJECT_ROOT / "ml" / "models" / "fraud_xgboost-v1.meta.json"

assert MODEL_PATH.exists(), "Modèle absent — lancer `make ml-train`"
model = joblib.load(MODEL_PATH)
meta = json.loads(META_PATH.read_text())

print(f"Version       : {meta['model_version']}")
print(f"Entraîné le   : {meta['trained_at']}")
print(f"Train / Test  : {meta['n_train']} / {meta['n_test']}")
print(f"MLflow run_id : {meta['mlflow_run_id']}")
print()
print("Métriques OFFLINE (à l'entraînement, sur le hold-out temporel) :")
for k, v in meta["metrics"].items():
    print(f"  {k:12s} = {v}")

# Le modèle est entraîné sur un ndarray brut (pas un DataFrame nommé), donc
# XGBoost ne connaît pas de noms de colonnes en interne. On vérifie plutôt :
# (1) le nombre de features attendu par le booster correspond à FEATURE_NAMES,
# (2) les métadonnées .meta.json (écrites par train_fraud_model.py au même
# moment que le modèle) sont alignées avec ml/features.py — c'est ce qui
# détecterait un mismatch colonne/valeur silencieux si features.py changeait
# sans réentraîner.
assert model.n_features_in_ == len(FEATURE_NAMES), (
    f"Le modèle attend {model.n_features_in_} features, "
    f"ml/features.FEATURE_NAMES en définit {len(FEATURE_NAMES)}"
)
assert meta["feature_names"] == FEATURE_NAMES, (
    "Incohérence entre les features déclarées dans .meta.json et ml/features.FEATURE_NAMES — "
    "risque de mismatch colonne/valeur silencieux à l'inférence"
)
print("\n[OK] Features du modèle alignées avec ml/features.py")


Version       : xgboost-v1
Entraîné le   : 2026-09-15T14:21:00.845781+00:00
Train / Test  : 55239 / 13810
MLflow run_id : 9755ccbec8054b049ccfe7d92f31dc4a

Métriques OFFLINE (à l'entraînement, sur le hold-out temporel) :
  precision    = 0.9414567733151804
  recall       = 0.965782122905028
  f1           = 0.953464322647363
  roc_auc      = 0.9968123734901757
  n_test       = 13810
  n_test_fraud = 2864

[OK] Features du modèle alignées avec ml/features.py


In [15]:
importances = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=True)
fig = px.bar(importances, x="importance", y="feature", orientation="h",
             title="Importance des features (XGBoost, gain)")
fig.show()


In [16]:
# Écart offline vs online : si l'online est significativement pire que
# l'offline malgré des features alignées (cellule précédente), la cause est
# un problème de SERVICE (skew, cache, feedback loop) et non le modèle
# lui-même — c'est exactement ce qu'a montré l'incident du 2026-09-15.
offline_precision = meta["metrics"]["precision"]
offline_recall = meta["metrics"]["recall"]
gap_p = offline_precision - live.get("precision", float("nan"))
gap_r = offline_recall - live.get("recall", float("nan"))
print(f"Écart précision (offline - online) : {gap_p:+.1%}")
print(f"Écart rappel    (offline - online) : {gap_r:+.1%}")
if abs(gap_p) > 0.15:
    print("[ALERTE] Écart offline/online important — investiguer la chaîne de service "
          "(Redis, cache modèle, doublons CDC) avant de suspecter le modèle.")
else:
    print("[OK] Performance online cohérente avec l'offline — pas de skew de service détecté.")


Écart précision (offline - online) : -1.4%
Écart rappel    (offline - online) : +3.0%
[OK] Performance online cohérente avec l'offline — pas de skew de service détecté.


## 6. Détection de drift (Evidently)

Compare la distribution des features **entre l'entraînement** (référence)
et **la fenêtre courante** — même logique que `ml/monitor.py::compute_drift`,
rejouée ici pour inspection visuelle détaillée (le monitoring en prod ne
donne qu'un score agrégé `drift_share`).


In [17]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Référence = fenêtre d'entraînement (recalcul SQL, indépendant de Redis).
# Actuelle = features servies récemment (Mongo, ce que le modèle a vraiment vu).
FEATURE_COLS = ["velocity_1h", "velocity_24h"]  # sous-ensemble comparable des 2 sources

ref = trained_v.rename(columns={"train_v1h": "velocity_1h", "train_v24h": "velocity_24h"})[FEATURE_COLS]
cur = served_v[FEATURE_COLS] if len(served_v) else pd.DataFrame(columns=FEATURE_COLS)

if len(cur) >= 30:
    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=ref, current_data=cur)
    report.show(mode="inline")
else:
    print("[SKIP] Pas assez de données servies récentes pour un rapport de drift fiable (< 30 lignes).")


## 7. Historique de monitoring (`ml_monitoring`, MongoDB)

Chaque cycle de `ml/monitor.py` (drift + performance servie) est journalisé
ici. On visualise précision/rappel/drift dans le temps avec les
déclenchements de retrain — l'incident du 2026-09-15 y est visible comme un
plateau de précision basse malgré des retrains répétés (cf. §4.3 pour
l'explication : le scorer ne rechargeait jamais le nouveau modèle).


In [18]:
history = list(mongo_db.ml_monitoring.find().sort("checked_at", 1))
hist_df = pd.DataFrame([{
    "checked_at": h["checked_at"],
    "status": h.get("status"),
    "precision": (h.get("performance") or {}).get("precision"),
    "recall": (h.get("performance") or {}).get("recall"),
    "drift_share": (h.get("drift") or {}).get("drift_share"),
    "retrain_triggered": h.get("retrain_triggered", False),
} for h in history])

if len(hist_df):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist_df["checked_at"], y=hist_df["precision"], name="Précision", mode="lines+markers"))
    fig.add_trace(go.Scatter(x=hist_df["checked_at"], y=hist_df["recall"], name="Rappel", mode="lines+markers"))
    retrains = hist_df[hist_df["retrain_triggered"]]
    fig.add_trace(go.Scatter(x=retrains["checked_at"], y=retrains["precision"], mode="markers",
                              marker=dict(size=12, symbol="star", color="red"), name="Retrain déclenché"))
    fig.add_hline(y=ML_MIN_PRECISION, line_dash="dot", annotation_text="seuil précision mini")
    fig.add_hline(y=ML_MIN_RECALL, line_dash="dot", annotation_text="seuil rappel mini")
    fig.update_layout(title="Historique monitoring ML — précision/rappel servis dans le temps")
    fig.show()
    display(hist_df.tail(10))
else:
    print("[SKIP] Aucun historique ml_monitoring — le service ml-monitor n'a pas encore tourné.")


,checked_at,status,precision,recall,drift_share,retrain_triggered
105,2026-09-15 14:04:33.123,alert,0.243938,0.973510,0.428571,False
106,2026-09-15 14:06:35.018,alert,0.235339,0.972816,0.428571,False
107,2026-09-15 14:08:37.105,alert,0.241903,0.974096,0.428571,True
108,2026-09-15 14:10:41.565,alert,0.245795,0.971897,0.428571,False
109,2026-09-15 14:12:44.431,alert,0.249857,0.971337,0.428571,False
110,2026-09-15 14:14:47.046,alert,0.264734,0.970142,0.428571,True
111,2026-09-15 14:16:52.015,alert,0.301081,0.954443,0.428571,False
112,2026-09-15 14:18:54.076,alert,0.313375,0.953184,0.428571,False
113,2026-09-15 14:20:56.080,alert,0.322329,0.949626,0.428571,True
114,2026-09-15 14:23:00.857,alert,0.349954,0.949698,0.428571,False


## 8. Synthèse

Relance cette cellule seule (`Restart & Run All` avant) pour un contrôle
rapide avant une démo ou la soutenance — elle échoue bruyamment (`AssertionError`)
si l'un des garde-fous ci-dessus n'est plus respecté.


In [19]:
print("=" * 60)
print("SYNTHÈSE AUDIT")
print("=" * 60)
print(f"Transactions orphelines (FK)         : 0 (vérifié §1)")
print(f"Doublons idempotency_key             : 0 (vérifié §1)")
print(f"txn_id dupliqués fraud_alerts (6h)    : {len(dup_txn)} (vérifié §2)")
if live.get("predicted_fraud") and live.get("actual_fraud"):
    print(f"Précision servie (fenêtre courante)   : {live['precision']:.1%}")
    print(f"Rappel servi (fenêtre courante)        : {live['recall']:.1%}")
print(f"Features modèle alignées ml/features.py : OK (vérifié §5)")
print("=" * 60)
print("Tous les garde-fous (`assert`) ci-dessus sont passés sans exception -> audit OK.")


SYNTHÈSE AUDIT
Transactions orphelines (FK)         : 0 (vérifié §1)
Doublons idempotency_key             : 0 (vérifié §1)
txn_id dupliqués fraud_alerts (6h)    : 0 (vérifié §2)
Précision servie (fenêtre courante)   : 95.5%
Rappel servi (fenêtre courante)        : 93.6%
Features modèle alignées ml/features.py : OK (vérifié §5)
Tous les garde-fous (`assert`) ci-dessus sont passés sans exception -> audit OK.
